In [4]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# estilo base
sns.set_theme(style="white")

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 1,
    "axes.linewidth": 0.8,
    "figure.dpi": 300
})

# 1 coluna
FIGSIZE_SINGLE = (7, 5)

# 2 colunas
FIGSIZE_DOUBLE = (7, 3)

#palette = sns.color_palette("colorblind")
palette = sns.color_palette("Set2") #("flare") #("Set2") #("husl", 9) #("pastel")

# pasta com os ficheiros
DATA_DIR = "./"

data = []

def parse_filename(filename):
    # remove prefixo e extensão
    name = filename.replace("results_", "").replace(".json", "")
    
    parts = name.split("_")
    
    attack = parts[0]
    type = parts[1]  # adversarial, random, etc.
    feature = None
    if len(parts) > 2:
        feature = parts[2]  # size, iat, both

    return attack, type, feature


# ler todos os ficheiros
# iterar sobre as pastas na pasta
for folder in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, folder)
    if os.path.isdir(folder_path):
        for folder2 in os.listdir(folder_path):
            folder2_path = os.path.join(folder_path, folder2)
            if os.path.isdir(folder2_path):
                for file in os.listdir(folder2_path):
                    if file.endswith(".json"):
                        with open(os.path.join(folder2_path, file)) as f:
                            content = json.load(f)
                
                        attack, type, feature = parse_filename(file)

                        if type == "original":
                            data.append({
                                "Attack": attack,
                                "Feature": type,
                                "Time Ratio": 1,
                                "Model": content["model"],
                                "ASR": content["metrics"]["attack_success_rate"]
                            })
                        else:
                            data.append({
                                "Attack": attack,
                                "Feature": feature,
                                "Time Ratio": content["Time_ratio"],
                                "Model": content["model"],
                                "ASR": content["metrics"]["attack_success_rate"]
                            })


df = pd.DataFrame(data)
# garantir ordem consistente
feature_order = ["original", "iat", "size", "both"]
df["Feature"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)

df["Feature"] = df["Feature"].map({
    "original": "Original",
    "iat": "IAT",
    "size": "Size",
    "both": "Both"
})

palette_fixed = {
    "Original": palette[0],
    "IAT": palette[1],
    "Size": palette[2],
    "Both": palette[3],
}

# garantir ordem consistente
attack_order = ["BruteForce", "DDoS", "PortScan"]
df["Attack"] = pd.Categorical(df["Attack"], categories=attack_order, ordered=True)

df["Attack"] = df["Attack"].map({
    "BruteForce": "Brute Force",
    "DDoS": "DDoS",
    "PortScan": "Port Scan",
})

palette_fixed_attack = {
    "Brute Force": palette[3],
    "DDoS": palette[2],
    "Port Scan": palette[1],
}

In [6]:
import matplotlib.gridspec as gridspec

attacks = df["Attack"].cat.categories
n = len(attacks)

fig = plt.figure(figsize=(12, 8))

# Criar grid: 2 linhas, 2 colunas
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

# Posições dos 3 subplots
axes = [
    fig.add_subplot(gs[0, 0]),  # topo esquerda
    fig.add_subplot(gs[0, 1]),  # topo direita
    fig.add_subplot(gs[1, :]),  # baixo, ocupando as 2 colunas (centrado)
]

# Ajustar o terceiro plot para ficar centrado e do mesmo tamanho
# Substituir gs[1, :] por uma posição manual centrada
axes[2].remove()
axes[2] = fig.add_axes([0.3, 0.08, 0.4, 0.35])  # [left, bottom, width, height]


for ax, attack in zip(axes, attacks):
    subset = df[df["Attack"] == attack]
    subset = subset[subset["Feature"].isin(["Original", "IAT"])]

    sns.lineplot(
        data=subset,
        x="Time Ratio",
        y="ASR",
        hue="Model",
        marker="o",
        palette=palette,
        errorbar=None,
        ax=ax
    )

    ax.set_title(attack)
    ax.set_xlabel("Time Ratio")
    ax.set_ylabel("ASR")
    ax.set_ylim(0, axes[0].get_ylim()[1])  # manter escala Y igual

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=6, padding=2)

    sns.despine(ax=ax)

# legenda global (1 só para todos os subplots)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="", frameon=False, loc="upper center", 
           bbox_to_anchor=(0.5, 1), ncol=len(labels))

# Remover legendas individuais
for ax in axes:
    if ax.get_legend():
        ax.get_legend().remove()

plt.savefig("plots/ieee_asr_ratio_all.png", bbox_inches="tight", dpi=300)
plt.close()

/tmp/user/1403/ipykernel_2438875/2053371785.py:28: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(
/tmp/user/1403/ipykernel_2438875/2053371785.py:28: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(
/tmp/user/1403/ipykernel_2438875/2053371785.py:28: UserWarning: The palette list has more values (8) than needed (5), which may not be intended.
  sns.lineplot(
